# Grid Dynamics Armenia Jobs Scraping

**ATS:** Custom WordPress + Angular widget  
**Method:** Playwright renders the page, extracts Yerevan job links, fetches each detail page  
**Why Playwright:** Job listings are loaded client-side by an Angular widget; the WordPress REST API only returns location filter config, not actual jobs  
**Outputs:** `data/raw/jobs/griddynamics_jobs_raw.csv`, `data/processed/jobs/griddynamics_jobs_standardized.csv`

In [1]:
from pathlib import Path, Path as P
import os

_nb = globals().get('__vsc_ipynb_file__') or globals().get('__file__', None)
if _nb:
    os.chdir(P(_nb).resolve().parent.parent.parent)
elif not (P.cwd() / 'data').exists():
    _r = P.cwd()
    for _ in range(4):
        if (_r / 'data').exists(): break
        _r = _r.parent
    os.chdir(_r)
assert (P.cwd() / 'data').exists(), f'Cannot find root from {P.cwd()}'
RAW_DIR  = P.cwd() / 'data/raw/jobs'
PROC_DIR = P.cwd() / 'data/processed/jobs'
print('Root:', P.cwd())

Root: /Users/lianaaghamalyan/thesis_data


In [2]:
import re, asyncio, pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE = 'https://www.griddynamics.com'
LISTING_URL = f'{BASE}/careers/discover-openings'

def html_to_text(h):
    if not h: return ''
    t = BeautifulSoup(str(h), 'html.parser').get_text('\n', strip=True)
    return re.sub(r'\n{3,}', '\n\n', t).strip()

print(f'Listing URL: {LISTING_URL}')

Listing URL: https://www.griddynamics.com/careers/discover-openings


In [3]:
async def scrape_griddynamics():
    records = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()

        # Step 1: Load listing page and collect Yerevan job links
        print('Loading listing page...')
        await page.goto(LISTING_URL, wait_until='networkidle', timeout=30000)
        await page.wait_for_timeout(3000)

        content = await page.content()
        soup = BeautifulSoup(content, 'html.parser')

        yerevan_links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            text = a.get_text(strip=True)
            if '/careers/vacancy/' in href and ('yerevan' in text.lower() or 'armenia' in text.lower()):
                full_url = BASE + href if href.startswith('/') else href
                # Parse title and location from combined link text
                # Text format: 'Job TitleYerevan, Armenia'
                loc_match = re.search(r'(Yerevan.*)', text, re.IGNORECASE)
                location = loc_match.group(1).strip() if loc_match else 'Yerevan, Armenia'
                title = text[:loc_match.start()].strip() if loc_match else text
                yerevan_links.append({'url': full_url, 'title': title, 'location': location})

        print(f'Yerevan jobs found: {len(yerevan_links)}')
        for j in yerevan_links:
            print(f"  {j['title']:<55} | {j['location']}")

        # Step 2: Fetch detail pages
        for i, job in enumerate(yerevan_links, 1):
            print(f'\n[{i}/{len(yerevan_links)}] {job["title"]}')
            try:
                await page.goto(job['url'], wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1500)
                detail_content = await page.content()
                detail_soup = BeautifulSoup(detail_content, 'html.parser')

                # Remove nav/footer boilerplate
                for tag in detail_soup.find_all(['nav', 'footer', 'header']):
                    tag.decompose()

                # Try to find main job content
                main = (detail_soup.find('main') or
                        detail_soup.find('article') or
                        detail_soup.find(class_=re.compile(r'vacancy|job-detail|content', re.I)))
                if not main:
                    main = detail_soup.find('body')

                full_text = re.sub(r'\n{3,}', '\n\n',
                                   main.get_text('\n', strip=True)).strip() if main else ''
                print(f'  {len(full_text)} chars')

                records.append({
                    'source': 'griddynamics',
                    'source_type': 'company_portal',
                    'source_url': job['url'],
                    'job_title': job['title'],
                    'company_name': 'Grid Dynamics',
                    'location': job['location'],
                    'department': '',
                    'employment_type': '',
                    'seniority_level': '',
                    'industries': 'Technology / IT Services',
                    'posting_date': '',
                    'skills_tags': '',
                    'full_text': full_text,
                })
            except Exception as e:
                print(f'  Error: {e}')
            await page.wait_for_timeout(1000)

        await browser.close()
    return records

records = await scrape_griddynamics()

Loading listing page...
Yerevan jobs found: 11
  Senior System Analyst                                   | Yerevan, Armenia
  Senior Big Data Engineer                                | Yerevan, Armenia
  Big Data Delivery Architect                             | Yerevan, Armenia
  Senior C++ Developer                                    | Yerevan, Armenia
  AI Architect                                            | Yerevan, Armenia
  Senior Machine Learning Engineer                        | Yerevan, Armenia
  Junior AP Accountant                                    | Yerevan, Armenia
  Staff Software Engineer (Java)                          | Yerevan, Armenia
  Senior Python Engineer – AI Transition Program          | Yerevan, Armenia
  Technical Sales Consultant (proficient in German and English) | Yerevan, Armenia
  Senior/Principal Search Engineer                        | Yerevan, Armenia

[1/11] Senior System Analyst
  5400 chars

[2/11] Senior Big Data Engineer
  4675 chars

[3/11] Big

In [4]:
df = pd.DataFrame(records)
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)
df.to_csv(RAW_DIR  / 'griddynamics_jobs_raw.csv', index=False)
df.to_csv(PROC_DIR / 'griddynamics_jobs_standardized.csv', index=False)
print(f'Saved {len(df)} rows')
if len(df):
    print(df[['job_title', 'location']].to_string())

Saved 11 rows
                                                        job_title          location
0                                           Senior System Analyst  Yerevan, Armenia
1                                        Senior Big Data Engineer  Yerevan, Armenia
2                                     Big Data Delivery Architect  Yerevan, Armenia
3                                            Senior C++ Developer  Yerevan, Armenia
4                                                    AI Architect  Yerevan, Armenia
5                                Senior Machine Learning Engineer  Yerevan, Armenia
6                                            Junior AP Accountant  Yerevan, Armenia
7                                  Staff Software Engineer (Java)  Yerevan, Armenia
8                  Senior Python Engineer – AI Transition Program  Yerevan, Armenia
9   Technical Sales Consultant (proficient in German and English)  Yerevan, Armenia
10                               Senior/Principal Search Engin